# Exploración de series — Cuenca Abierta

Este notebook carga las series generadas en `salida/`, resume su contenido y las visualiza con pandas y matplotlib. Para regenerarlas, cambia `EJECUTAR_GENERADOR` a `True` en la celda siguiente.

En VS Code, selecciona como kernel el Python del proyecto: `.venv\Scripts\python.exe`. Ejecuta las celdas en orden. Las gráficas son exploratorias; consulta la columna `origen` antes de interpretar una lectura como medida real.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "generar.py").exists():
    raise FileNotFoundError(
        f"No encuentro generar.py en {PROJECT_DIR}. Abre el notebook desde la raíz del proyecto."
    )
SALIDA_DIR = PROJECT_DIR / "salida"
plt.style.use("seaborn-v0_8-whitegrid")
print(f"Proyecto: {PROJECT_DIR}")
print(f"Python: {sys.executable}")

In [ ]:
# Opcional: volver a ejecutar toda la generación (puede tardar unos minutos).
EJECUTAR_GENERADOR = False

if EJECUTAR_GENERADOR:
    subprocess.run([sys.executable, str(PROJECT_DIR / "generar.py")], cwd=PROJECT_DIR, check=True)
else:
    print("Se usarán los archivos existentes en salida/.")

In [ ]:
def leer_csv(nombre, **kwargs):
    ruta = SALIDA_DIR / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}. Ejecuta generar.py o activa EJECUTAR_GENERADOR.")
    return pd.read_csv(ruta, **kwargs)


caudal = leer_csv("caudal_diario.csv")
caudal["timestamp_utc"] = pd.to_datetime(caudal["timestamp_utc"], utc=True, errors="coerce")
caudal["valor"] = pd.to_numeric(caudal["valor"], errors="coerce")

semaforo = leer_csv("semaforo_diario.csv")
semaforo["fecha"] = pd.to_datetime(semaforo["fecha"], errors="coerce")

sensores_path = SALIDA_DIR / "lecturas_sensores.parquet"
sensores = pd.read_parquet(sensores_path) if sensores_path.exists() else None
if sensores is not None:
    sensores["timestamp_utc"] = pd.to_datetime(sensores["timestamp_utc"], utc=True, errors="coerce")
    sensores["valor"] = pd.to_numeric(sensores["valor"], errors="coerce")

laboratorio_path = SALIDA_DIR / "muestreos_laboratorio.csv"
laboratorio = leer_csv("muestreos_laboratorio.csv") if laboratorio_path.exists() else None
if laboratorio is not None:
    laboratorio["timestamp_utc"] = pd.to_datetime(laboratorio["timestamp_utc"], utc=True, errors="coerce")
    laboratorio["valor"] = pd.to_numeric(laboratorio["valor"], errors="coerce")

print(f"Caudal diario: {caudal.shape[0]:,} filas x {caudal.shape[1]} columnas")
print(f"Semáforo diario: {semaforo.shape[0]:,} filas x {semaforo.shape[1]} columnas")
print(f"Sensores: {sensores.shape[0]:,} filas x {sensores.shape[1]} columnas" if sensores is not None else "Sensores: archivo Parquet no disponible")
print(f"Laboratorio: {laboratorio.shape[0]:,} filas x {laboratorio.shape[1]} columnas" if laboratorio is not None else "Laboratorio: archivo CSV no disponible")

In [ ]:
for nombre, df, fecha_col in [
    ("Caudal diario", caudal, "timestamp_utc"),
    ("Sensores", sensores, "timestamp_utc"),
    ("Laboratorio", laboratorio, "timestamp_utc"),
    ("Semáforo diario", semaforo, "fecha"),
]:
    if df is None:
        continue
    fechas = df[fecha_col].dropna()
    rango = f"{fechas.min()} — {fechas.max()}" if not fechas.empty else "sin fechas válidas"
    print(f"\n{nombre}: {rango}")
    print("Columnas:", ", ".join(df.columns))

display(caudal.head())
if sensores is not None:
    display(sensores.head())
if laboratorio is not None:
    display(laboratorio.head())

In [ ]:
# Caudal diario por punto de monitoreo.
if {"timestamp_utc", "valor", "punto_id"}.issubset(caudal.columns):
    caudal_plot = caudal.copy()
    if "variable" in caudal_plot.columns:
        caudal_plot = caudal_plot[caudal_plot["variable"] == "caudal_medio_diario"]
    fig, ax = plt.subplots(figsize=(13, 5))
    for punto, grupo in caudal_plot.groupby("punto_id"):
        grupo = grupo.sort_values("timestamp_utc")
        ax.plot(grupo["timestamp_utc"], grupo["valor"], label=punto, linewidth=0.8)
    ax.set(title="Caudal medio diario por punto", xlabel="Fecha (UTC)", ylabel="Caudal (m³/s)")
    ax.legend(title="Punto")
    fig.tight_layout()
    plt.show()
else:
    print("No se encontraron las columnas necesarias para graficar el caudal.")

In [ ]:
# Cambia estas opciones para explorar otra variable, punto o ventana temporal.
if sensores is not None and {"timestamp_utc", "valor", "variable"}.issubset(sensores.columns):
    variables_sensor = sorted(sensores["variable"].dropna().unique().tolist())
    puntos_sensor = sorted(sensores["punto_id"].dropna().unique().tolist()) if "punto_id" in sensores.columns else []
    SENSOR_VARIABLE = "temperatura" if "temperatura" in variables_sensor else (variables_sensor[0] if variables_sensor else None)
    SENSOR_PUNTO = puntos_sensor[0] if puntos_sensor else None  # Usa None para incluir todos los puntos.
    DIAS_A_MOSTRAR = 7

    print("Variables disponibles:", variables_sensor)
    print("Puntos disponibles:", puntos_sensor)
    if SENSOR_VARIABLE is None:
        print("No hay variables de sensor para graficar.")
    else:
        fecha_fin = sensores["timestamp_utc"].max()
        fecha_inicio = fecha_fin - pd.Timedelta(days=DIAS_A_MOSTRAR)
        sensor_plot = sensores[
            sensores["variable"].eq(SENSOR_VARIABLE)
            & sensores["timestamp_utc"].between(fecha_inicio, fecha_fin)
        ]
        if SENSOR_PUNTO is not None and "punto_id" in sensor_plot.columns:
            sensor_plot = sensor_plot[sensor_plot["punto_id"].eq(SENSOR_PUNTO)]
        fig, ax = plt.subplots(figsize=(13, 5))
        if "punto_id" in sensor_plot.columns:
            for punto, grupo in sensor_plot.groupby("punto_id"):
                grupo = grupo.sort_values("timestamp_utc")
                ax.plot(grupo["timestamp_utc"], grupo["valor"], label=punto, linewidth=0.8)
            ax.legend(title="Punto")
        else:
            sensor_plot = sensor_plot.sort_values("timestamp_utc")
            ax.plot(sensor_plot["timestamp_utc"], sensor_plot["valor"], linewidth=0.8)
        ax.set(title=f"Sensor: {SENSOR_VARIABLE} — últimos {DIAS_A_MOSTRAR} días", xlabel="Fecha (UTC)", ylabel=SENSOR_VARIABLE)
        fig.tight_layout()
        plt.show()
else:
    print("No se encontraron lecturas de sensores con las columnas esperadas.")

In [ ]:
# Conteo diario de colores del semáforo (se excluye la fila-resumen por variable).
if {"fecha", "color"}.issubset(semaforo.columns):
    sem_plot = semaforo.copy()
    if "variable" in sem_plot.columns:
        sem_plot = sem_plot[sem_plot["variable"] == "_resumen"]
    conteo = sem_plot.groupby(["fecha", "color"]).size().unstack(fill_value=0)
    if not conteo.empty:
        colores = {"verde": "#2e8b57", "amarillo": "#e5b700", "rojo": "#d62728", "gris": "#999999"}
        fig, ax = plt.subplots(figsize=(13, 5))
        conteo.plot.area(ax=ax, stacked=True, color=[colores.get(c, "#4c78a8") for c in conteo.columns], alpha=0.8)
        ax.set(title="Semáforo diario: cantidad de puntos/usos por color", xlabel="Fecha", ylabel="Cantidad")
        fig.tight_layout()
        plt.show()
    else:
        print("No hay filas de resumen de semáforo para graficar.")
else:
    print("No se encontraron fecha y color en el semáforo diario.")

In [ ]:
# Distribución temporal de una variable de laboratorio.
if laboratorio is not None and {"timestamp_utc", "valor", "variable"}.issubset(laboratorio.columns):
    variables_lab = sorted(laboratorio["variable"].dropna().unique().tolist())
    LAB_VARIABLE = "arsenico" if "arsenico" in variables_lab else (variables_lab[0] if variables_lab else None)
    print("Variables disponibles:", variables_lab)
    if LAB_VARIABLE is not None:
        lab_plot = laboratorio[laboratorio["variable"].eq(LAB_VARIABLE)].dropna(subset=["valor"])
        fig, ax = plt.subplots(figsize=(11, 4))
        if "punto_id" in lab_plot.columns:
            for punto, grupo in lab_plot.groupby("punto_id"):
                ax.scatter(grupo["timestamp_utc"], grupo["valor"], label=punto, alpha=0.75)
            ax.legend(title="Punto")
        else:
            ax.scatter(lab_plot["timestamp_utc"], lab_plot["valor"], alpha=0.75)
        ax.set(title=f"Muestreos de laboratorio: {LAB_VARIABLE}", xlabel="Fecha (UTC)", ylabel=LAB_VARIABLE)
        fig.tight_layout()
        plt.show()
else:
    print("No se encontraron las columnas necesarias para graficar laboratorio.")